In [ ]:
%run "./00_CC_Mapping_Setup.ipynb" 

In [ ]:
%sql
/* ===================================================================================
   METRIC: TP01 - TP Jurisdictions
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.ra_adido_2025.fy25_list_of_units
   2. TP Unit Mapping: hive_metastore.ra_adido_2025.third_party_unit_mapping
   3. Country Risk Ratings: hive_metastore.ra_adido_2025.td_country_risk_rating_abac
   4. Third Party Data: hive_metastore.ra_adido_2025.abac_request_paul_v3
   
   BUSINESS QUESTION: 
   What number of third party relationships with third parties are in jurisdictions 
   considered a high risk for bribery and corruption during the assessment period?
   
   LOGIC SUMMARY:
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
      Only AUs in this filtered list (Canada, HK, Barbados + US_OR_CANADA = 'CANADA') 
      will appear in the final report.
   2. HIGH RISK EVALUATION: Checks three columns against the ABAC High Risk list. 
      If ALL THREE are blank/null, automatically defaults to High Risk.
   3. TPRM STRING MAPPING: Maps TP engagements to AU IDs by checking if the 
      'TPRM_Units' string exists inside the 'owning_lob' or 'lob_using' columns.
      - Blocks blank mapping strings using NULLIF to prevent wildcard explosions.
      - Uses Databricks RLIKE with word boundaries (\b) for exact phrase matching.
   4. AGGREGATING: Counts DISTINCT EngagementNumbers per mapped AU to remove duplicates.
   5. FINAL OUTPUT: Strict 6-column structure, converting NULL sums to '0'.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data: This is our strict anchor for the final output
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

High_Risk_Countries AS (
    -- 2. Grab high-risk jurisdictions from the ABAC rating table
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE TRIM(FinalABACRating) = 'High'
),

Third_Party_Risk_Data AS (
    -- 3. Extract TP data and flag if jurisdiction is completely missing
    SELECT 
        EngagementNumber,
        owning_lob,
        lob_using,
        location_of_tp,
        country_prod_serv_originates,
        Td_Countries_Impacted,
        -- "If no Jurisdiction, then automatically high risk"
        CASE WHEN COALESCE(TRIM(location_of_tp), '') = ''
              AND COALESCE(TRIM(country_prod_serv_originates), '') = ''
              AND COALESCE(TRIM(Td_Countries_Impacted), '') = '' THEN 1
             ELSE 0 END AS Is_Missing_Jurisdiction
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3
    WHERE EngagementNumber IS NOT NULL
),

High_Risk_Engagements AS (
    -- 4. Filter for engagements that touch a High Risk country or have NO country
    SELECT DISTINCT 
        tp.EngagementNumber,
        tp.owning_lob,
        tp.lob_using
    FROM Third_Party_Risk_Data tp
    LEFT JOIN High_Risk_Countries h1 ON UPPER(TRIM(tp.location_of_tp)) = h1.CountryName
    LEFT JOIN High_Risk_Countries h2 ON UPPER(TRIM(tp.country_prod_serv_originates)) = h2.CountryName
    LEFT JOIN High_Risk_Countries h3 ON UPPER(TRIM(tp.Td_Countries_Impacted)) = h3.CountryName
    WHERE h1.CountryName IS NOT NULL
       OR h2.CountryName IS NOT NULL
       OR h3.CountryName IS NOT NULL
       OR tp.Is_Missing_Jurisdiction = 1
),

Engagements_By_AU AS (
    -- 5. Map high-risk engagements using the TPRM unit mapping table
    SELECT 
        TRIM(CAST(map.Assessable_Unit_ID AS STRING)) AS Mapped_AU_ID,
        COUNT(DISTINCT hr.EngagementNumber) AS High_Risk_Count
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping map
    
    -- Uses Regex word boundaries and completely blocks blank mapping strings
    INNER JOIN High_Risk_Engagements hr
        ON NULLIF(TRIM(map.TPRM_Units), '') IS NOT NULL
       AND (
           UPPER(hr.owning_lob) LIKE '%' || UPPER(TRIM(map.TPRM_Units)) || '%'
            OR UPPER(hr.lob_using) LIKE '%' || UPPER(TRIM(map.TPRM_Units)) || '%'
       )
    WHERE map.Assessable_Unit_ID IS NOT NULL
    GROUP BY TRIM(CAST(map.Assessable_Unit_ID AS STRING))
)

-- 6. Final Template: Strict 6-column output anchored to Master AU List
SELECT 
    a.BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'TP01' AS QuestionID,               
    COALESCE(CAST(e.High_Risk_Count AS STRING), '0') AS Response,
    a.In_ABAC_AU_List
    
FROM Master_AUs a
LEFT JOIN Engagements_By_AU e 
    ON a.BusinessID = e.Mapped_AU_ID
ORDER BY a.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: TP01 Drill-Down
   
   PURPOSE: Shows EVERY high-risk third-party engagement, regardless of whether 
   the string mapped to an AU, or whether that AU exists in the Master List. 
   Useful for tracking unmapped high-risk engagements and fixing regex matches.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

High_Risk_Countries AS (
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE TRIM(FinalABACRating) = 'High'
),

Third_Party_Risk_Data AS (
    SELECT 
        EngagementNumber,
        owning_lob,
        lob_using,
        location_of_tp,
        country_prod_serv_originates,
        Td_Countries_Impacted,
        CASE WHEN COALESCE(TRIM(location_of_tp), '') = ''
              AND COALESCE(TRIM(country_prod_serv_originates), '') = ''
              AND COALESCE(TRIM(Td_Countries_Impacted), '') = '' THEN 1
             ELSE 0 END AS Is_Missing_Jurisdiction
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3
    WHERE EngagementNumber IS NOT NULL
),

High_Risk_Engagements AS (
    SELECT DISTINCT 
        tp.EngagementNumber,
        tp.owning_lob,
        tp.lob_using,
        tp.location_of_tp,
        tp.country_prod_serv_originates,
        tp.Td_Countries_Impacted,
        tp.Is_Missing_Jurisdiction,
        -- Flag to show exactly which column triggered the risk
        CASE 
            WHEN tp.Is_Missing_Jurisdiction = 1 THEN 'Missing All Jurisdictions'
            WHEN h1.CountryName IS NOT NULL THEN 'location_of_tp'
            WHEN h2.CountryName IS NOT NULL THEN 'country_prod_serv_originates'
            WHEN h3.CountryName IS NOT NULL THEN 'Td_Countries_Impacted'
        END AS Risk_Trigger
    FROM Third_Party_Risk_Data tp
    LEFT JOIN High_Risk_Countries h1 ON UPPER(TRIM(tp.location_of_tp)) = h1.CountryName
    LEFT JOIN High_Risk_Countries h2 ON UPPER(TRIM(tp.country_prod_serv_originates)) = h2.CountryName
    LEFT JOIN High_Risk_Countries h3 ON UPPER(TRIM(tp.Td_Countries_Impacted)) = h3.CountryName
    WHERE h1.CountryName IS NOT NULL
       OR h2.CountryName IS NOT NULL
       OR h3.CountryName IS NOT NULL
       OR tp.Is_Missing_Jurisdiction = 1
)

SELECT DISTINCT
    COALESCE(TRIM(CAST(map.Assessable_Unit_ID AS STRING)), 'UNMAPPED_ENGAGEMENT') AS BusinessID,
    COALESCE(mast.In_ABAC_AU_List, 'No') AS In_ABAC_AU_List,
    hr.EngagementNumber,
    hr.owning_lob AS Raw_Col_K_owning_lob,
    hr.lob_using AS Raw_Col_L_lob_using,
    map.TPRM_Units AS Matched_Mapping_String,
    hr.Risk_Trigger
FROM High_Risk_Engagements hr
-- Join to mapping table using the RLIKE logic
LEFT JOIN hive_metastore.ra_adido_2025.third_party_unit_mapping map
    ON NULLIF(TRIM(map.TPRM_Units), '') IS NOT NULL
   AND (
       UPPER(hr.owning_lob) LIKE '%' || UPPER(TRIM(map.TPRM_Units)) || '%'
       OR UPPER(hr.lob_using) LIKE '%' || UPPER(TRIM(map.TPRM_Units)) || '%'
   )
LEFT JOIN Master_AUs mast
    ON TRIM(CAST(map.Assessable_Unit_ID AS STRING)) = mast.BusinessID
ORDER BY BusinessID, hr.EngagementNumber;

In [ ]:
%sql
/* DIAGNOSTIC 1: String Mapping Test */
SELECT 
    tp.EngagementNumber, 
    tp.owning_lob, 
    tp.lob_using, 
    map.TPRM_Units AS Attempted_Match_String, 
    map.Assessable_Unit_ID
FROM hive_metastore.ra_adido_2025.abac_request_paul_v3 tp
-- Try to join using the fuzzy text match
INNER JOIN hive_metastore.ra_adido_2025.third_party_unit_mapping map
    ON UPPER(tp.owning_lob) LIKE '%' || UPPER(TRIM(map.TPRM_Units)) || '%'
    OR UPPER(tp.lob_using) LIKE '%' || UPPER(TRIM(map.TPRM_Units)) || '%'
LIMIT 100;

In [ ]:
%sql
/* DIAGNOSTIC 2: High Risk Country spelling test */
WITH High_Risk_Countries AS (
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE TRIM(FinalABACRating) = 'High'
)
SELECT 
    tp.EngagementNumber, 
    tp.location_of_tp AS Raw_TP_Country, 
    h.CountryName AS Matched_ABAC_Country
FROM hive_metastore.ra_adido_2025.abac_request_paul_v3 tp
LEFT JOIN High_Risk_Countries h 
    ON UPPER(TRIM(tp.location_of_tp)) = h.CountryName
-- Only show rows where a country was actually listed
WHERE tp.location_of_tp IS NOT NULL
LIMIT 100;

In [ ]:
%sql
/* DIAGNOSTIC 3: Master AU Filter Test */
SELECT 
    BusinessID, 
    Business_Operational_Unit_Name, 
    Jurisdiction
FROM hive_metastore.ra_adido_2025.fy25_list_of_units
WHERE UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS');

In [ ]:
%sql
/* ===================================================================================
   DIAGNOSTIC 4: Orphaned Engagement Check
   
   PURPOSE: Looks specifically at the 4 'India' engagements to see what text is 
   inside their LOB columns and whether they successfully join to the mapping table.
=================================================================================== */

SELECT 
    tp.EngagementNumber,
    tp.location_of_tp AS Country,
    tp.owning_lob,
    tp.lob_using,
    map.TPRM_Units AS Attempted_Map_String,
    map.Assessable_Unit_ID AS Mapped_AU_ID
    
FROM hive_metastore.ra_adido_2025.abac_request_paul_v3 tp

-- LEFT JOIN so we can see the engagements even if the mapping fails
LEFT JOIN hive_metastore.ra_adido_2025.third_party_unit_mapping map
    ON UPPER(tp.owning_lob) LIKE '%' || UPPER(TRIM(map.TPRM_Units)) || '%'
    OR UPPER(tp.lob_using) LIKE '%' || UPPER(TRIM(map.TPRM_Units)) || '%'

-- Hardcoded the 4 engagements from your Diagnostic 2 screenshot
WHERE tp.EngagementNumber IN ('E41783', 'E24498', 'E43497', 'E41726');